# 02 · Data Cleaning
**Project:** EV Charging Demand & Cost Analytics + Predictive System  
**Input:** `data/raw/ev_charging_patterns.csv` (1,320 rows × 20 columns)  
**Output:** `data/processed/ev_charging_cleaned.csv`

Builds on findings from `01_data_understanding.ipynb`.  
**Rule:** The raw CSV is never modified — all operations are in-memory.

---

### Cleaning Roadmap
| Step | Issue | Action |
|------|--------|--------|
| 1 | Column name encoding artefact (`Temperature`) | Rename |
| 2 | 66 missing values in 3 numeric columns | Grouped median imputation |
| 3 | SoC Start/End out of [0, 100] range | Clip to valid bounds |
| 4 | SoC End < SoC Start (268 rows) | Flag + investigate → impute End SoC |
| 5 | Temperature > 50°C (physically implausible) | Clip to plausible range |
| 6 | Energy Consumed / Duration ≤ 0 | Validate (none expected) |
| 7 | Duplicate rows / User+Timestamp duplicates | Check + drop if any |
| 8 | Categorical text standardisation | Strip, title-case |
| 9 | Save cleaned dataset | Export to `/data/processed/` |

---

## 0 · Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 60)

RAW_PATH     = '../data/raw/ev_charging_patterns.csv'
CLEANED_PATH = '../data/processed/ev_charging_cleaned.csv'

print('Libraries loaded ')

Libraries loaded 


In [2]:
# Load raw data — never modify this object
df_raw = pd.read_csv(RAW_PATH)
# Working copy for all cleaning operations
df = df_raw.copy()

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Columns:', df.columns.tolist())

Loaded: 1,320 rows × 20 columns
Columns: ['User ID', 'Vehicle Model', 'Battery Capacity (kWh)', 'Charging Station ID', 'Charging Station Location', 'Charging Start Time', 'Charging End Time', 'Energy Consumed (kWh)', 'Charging Duration (hours)', 'Charging Rate (kW)', 'Charging Cost (USD)', 'Time of Day', 'Day of Week', 'State of Charge (Start %)', 'State of Charge (End %)', 'Distance Driven (since last charge) (km)', 'Temperature (°C)', 'Vehicle Age (years)', 'Charger Type', 'User Type']


## 0b · Before/After Progress Tracker
We maintain a running log of rows removed and nulls resolved at each step.

In [3]:
log = []   # Each entry: {'Step': ..., 'Action': ..., 'Rows': ..., 'Nulls': ...}

def snapshot(step_name, action_desc, df_current):
    """Record a before/after snapshot row."""
    log.append({
        'Step'  : step_name,
        'Action': action_desc,
        'Rows'  : len(df_current),
        'Nulls' : int(df_current.isnull().sum().sum())
    })

def show_log():
    summary = pd.DataFrame(log)
    summary['Rows Δ']  = summary['Rows'].diff().fillna(0).astype(int)
    summary['Nulls Δ'] = summary['Nulls'].diff().fillna(0).astype(int)
    display(summary)

# Baseline snapshot
snapshot('Baseline', 'Raw data loaded', df)
print('Tracker initialised.')

Tracker initialised.


---
## Step 1 · Fix Column Name Encoding Artefact

The `Temperature` column was stored with a Windows-1252 encoded degree sign (`°`) that appears
as a garbled character (`�`) when read in UTF-8. We rename it to a clean ASCII-safe name.

In [4]:
# Find the temperature column regardless of encoding
temp_col_old = [c for c in df.columns if 'emp' in c.lower() and c != 'Charging Start Time'][0]
print(f'Found temperature column: {repr(temp_col_old)}')

df.rename(columns={temp_col_old: 'Temperature (C)'}, inplace=True)

print('Renamed to: "Temperature (C)"')
snapshot('Step 1', 'Renamed temperature column', df)

Found temperature column: 'Temperature (°C)'
Renamed to: "Temperature (C)"


---
## Step 2 · Missing Value Audit & Imputation

### Findings from Step 01
Three columns have exactly **66 missing values each (~5% of 1,320 rows)**:
- `Energy Consumed (kWh)`
- `Charging Rate (kW)`
- `Distance Driven (since last charge) (km)`

### Imputation Strategy & Justification

**Why not drop rows?**  
Dropping 66 rows (5%) would bias the dataset — missing records are unlikely to be perfectly
random, and we'd lose information about certain charger types or user patterns.

**Why median over mean?**  
Energy and charging rate distributions are right-skewed (heavy users inflate the mean).
Median is more robust to outliers and preserves the central tendency better.

**Why group by Charger Type?**  
Energy delivered and charging rate are physically determined by charger hardware:
- DC Fast Charger: 50–350 kW → high energy, short duration  
- Level 2: 7–22 kW → moderate  
- Level 1: 1.4–1.9 kW → slow, long sessions  
Imputing within each charger group avoids mixing fundamentally different physics.

**Distance Driven:** Not charger-dependent (it reflects how far the car drove, not how it charged).
We use global median, optionally grouped by User Type to capture commuter vs. casual patterns.

In [5]:
# Full null audit before imputation
null_counts = df.isnull().sum()
null_pct    = (df.isnull().mean() * 100).round(2)

null_df = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
null_df = null_df[null_df['Null Count'] > 0]

print('=== Columns with Missing Values (before imputation) ===')
display(null_df)
print(f'\nTotal nulls: {df.isnull().sum().sum()}')

=== Columns with Missing Values (before imputation) ===


,Null Count,Null %
Energy Consumed (kWh),66,5.0000
Charging Rate (kW),66,5.0000
Distance Driven (since last charge) (km),66,5.0000



Total nulls: 198


In [6]:
# ── Impute Energy Consumed and Charging Rate — grouped by Charger Type ──────────
print('Charger Type group medians BEFORE imputation:')
print(df.groupby('Charger Type')[['Energy Consumed (kWh)', 'Charging Rate (kW)']].median())

for col in ['Energy Consumed (kWh)', 'Charging Rate (kW)']:
    group_medians = df.groupby('Charger Type')[col].transform('median')
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna(group_medians)
    print(f'\n  "{col}": filled {n_filled} nulls with Charger Type group median')
    print(f'  Remaining nulls: {df[col].isna().sum()}')

snapshot('Step 2a', 'Imputed Energy & Rate (Charger Type group median)', df)

Charger Type group medians BEFORE imputation:
                 Energy Consumed (kWh)  Charging Rate (kW)
Charger Type                                              
DC Fast Charger                40.2484             26.1352
Level 1                        42.2855             24.8458
Level 2                        46.0940             25.8578

  "Energy Consumed (kWh)": filled 66 nulls with Charger Type group median
  Remaining nulls: 0

  "Charging Rate (kW)": filled 66 nulls with Charger Type group median
  Remaining nulls: 0


In [7]:
# ── Impute Distance Driven — grouped by User Type ───────────────────────────────
dist_col = 'Distance Driven (since last charge) (km)'
print('User Type group medians for Distance Driven:')
print(df.groupby('User Type')[dist_col].median())

group_medians_dist = df.groupby('User Type')[dist_col].transform('median')
n_filled = df[dist_col].isna().sum()
df[dist_col] = df[dist_col].fillna(group_medians_dist)

# Fallback: global median for any remaining (e.g., if a User Type group has all-NaN)
if df[dist_col].isna().any():
    global_med = df[dist_col].median()
    df[dist_col] = df[dist_col].fillna(global_med)
    print(f'  Fallback global median applied.')

print(f'  "{dist_col}": filled {n_filled} nulls with User Type group median')
print(f'  Remaining nulls: {df[dist_col].isna().sum()}')

snapshot('Step 2b', 'Imputed Distance Driven (User Type group median)', df)

User Type group medians for Distance Driven:
User Type
Casual Driver            132.9596
Commuter                 161.1961
Long-Distance Traveler   157.3182
Name: Distance Driven (since last charge) (km), dtype: float64
  "Distance Driven (since last charge) (km)": filled 66 nulls with User Type group median
  Remaining nulls: 0


In [8]:
# Verify zero nulls remain
total_nulls = df.isnull().sum().sum()
print(f'Total nulls remaining: {total_nulls}')
assert total_nulls == 0, f'Unexpected nulls still present: {total_nulls}'
print(' All missing values resolved.')

Total nulls remaining: 0
 All missing values resolved.


---
## Step 3 · Validate & Fix State of Charge (SoC) Ranges

### Rules
- SoC (Start %) and SoC (End %) must be in **[0, 100]**
- SoC End should generally be **≥ SoC Start** (charging adds energy)

### Strategy for out-of-range SoC values
**Clip to [0, 100]:** Values like 101.3% are almost certainly measurement rounding errors
(sensor over-reads). Clipping is safer than dropping — the session itself is valid.

### Strategy for SoC End < SoC Start (268 rows)
This is physically implausible for a normal charging session. Possible explanations:
1. Data entry error (start/end swapped)
2. Session interrupted / battery used during charge (vehicle-to-grid)
3. Sensor glitch

**Decision:** We do NOT drop these rows (268 = 20% of data — too much to lose).
Instead we **swap Start/End SoC** when End < Start AND both are within [0,100],
then flag an `soc_swapped` indicator column for downstream model awareness.

In [9]:
soc_start = 'State of Charge (Start %)'
soc_end   = 'State of Charge (End %)'

# ── 3a. Report violations before fixing ─────────────────────────────────────────
print('=== SoC Range Violations (BEFORE fixing) ===')
print(f'  SoC Start < 0  : {(df[soc_start] < 0).sum()} rows')
print(f'  SoC Start > 100: {(df[soc_start] > 100).sum()} rows')
print(f'  SoC End   < 0  : {(df[soc_end] < 0).sum()} rows')
print(f'  SoC End   > 100: {(df[soc_end] > 100).sum()} rows')

# Show extreme out-of-range examples
oob = df[(df[soc_start] > 100) | (df[soc_end] > 100) |
         (df[soc_start] < 0)   | (df[soc_end] < 0)]
print(f'\n  Total out-of-range rows: {len(oob)}')
if len(oob) > 0:
    display(oob[[soc_start, soc_end, 'Energy Consumed (kWh)', 'Charger Type']].head(10))

=== SoC Range Violations (BEFORE fixing) ===
  SoC Start < 0  : 0 rows
  SoC Start > 100: 9 rows
  SoC End   < 0  : 0 rows
  SoC End   > 100: 23 rows

  Total out-of-range rows: 32


,State of Charge (Start %),State of Charge (End %),Energy Consumed (kWh),Charger Type
19,12.7549,116.0957,48.4825,Level 2
44,15.2209,132.9520,46.0940,Level 2
88,105.0322,83.0235,33.0729,Level 2
125,34.6002,101.6761,6.1890,Level 1
191,152.4898,68.1288,55.1888,Level 1
282,109.2658,82.3338,20.3535,Level 2
287,81.5593,177.7087,13.3190,Level 2
325,32.1489,113.2111,17.8905,Level 2
346,125.0872,97.0590,29.2984,Level 2
400,88.3173,103.0635,66.1891,DC Fast Charger


In [10]:
# ── 3b. Clip SoC columns to [0, 100] ────────────────────────────────────────────
df[soc_start] = df[soc_start].clip(0, 100)
df[soc_end]   = df[soc_end].clip(0, 100)

print('After clipping to [0, 100]:')
print(f'  SoC Start range: [{df[soc_start].min():.2f}, {df[soc_start].max():.2f}]')
print(f'  SoC End   range: [{df[soc_end].min():.2f}, {df[soc_end].max():.2f}]')

snapshot('Step 3a', 'Clipped SoC Start/End to [0, 100]', df)

After clipping to [0, 100]:
  SoC Start range: [2.33, 100.00]
  SoC End   range: [7.60, 100.00]


In [11]:
# ── 3c. Handle SoC End < SoC Start ──────────────────────────────────────────────
n_reversed = (df[soc_end] < df[soc_start]).sum()
print(f'Rows with SoC End < SoC Start: {n_reversed} ({n_reversed/len(df)*100:.1f}%)')

# Swap Start and End SoC for reversed rows
reversed_mask = df[soc_end] < df[soc_start]
df.loc[reversed_mask, [soc_start, soc_end]] = (
    df.loc[reversed_mask, [soc_end, soc_start]].values
)

# Add indicator flag
df['soc_was_swapped'] = reversed_mask.astype(int)

# Verify
still_reversed = (df[soc_end] < df[soc_start]).sum()
print(f'\nAfter swap:')
print(f'  Rows still with SoC End < SoC Start: {still_reversed}')
print(f'  Rows flagged with soc_was_swapped=1: {df["soc_was_swapped"].sum()}')
print(f'\nSoC delta (End - Start) stats after fix:')
display((df[soc_end] - df[soc_start]).describe().to_frame('SoC Delta (pp)'))

snapshot('Step 3b', 'Swapped reversed SoC pairs; added soc_was_swapped flag', df)

Rows with SoC End < SoC Start: 268 (20.3%)

After swap:
  Rows still with SoC End < SoC Start: 0
  Rows flagged with soc_was_swapped=1: 268

SoC delta (End - Start) stats after fix:


,SoC Delta (pp)
count,1320.0000
mean,31.6379
std,22.1947
min,0.0008
25%,12.8555
50%,27.0445
75%,47.7218
max,92.8257


---
## Step 4 · Validate Energy Consumed & Charging Duration

Both must be strictly positive (a session with zero or negative energy/duration is nonsensical).
**Strategy:** Flag any violations; drop only if confirmed impossible.

In [12]:
energy_col   = 'Energy Consumed (kWh)'
duration_col = 'Charging Duration (hours)'

n_energy_bad   = (df[energy_col]   <= 0).sum()
n_duration_bad = (df[duration_col] <= 0).sum()

print(f'Energy Consumed <= 0  : {n_energy_bad} rows')
print(f'Charging Duration <= 0: {n_duration_bad} rows')

if n_energy_bad == 0 and n_duration_bad == 0:
    print('\n All Energy and Duration values are strictly positive — no action needed.')
else:
    bad = df[(df[energy_col] <= 0) | (df[duration_col] <= 0)]
    print('\n️  Problematic rows:')
    display(bad[[energy_col, duration_col, 'Charger Type', 'User Type']])
    # Drop rows with physically impossible values
    df = df[(df[energy_col] > 0) & (df[duration_col] > 0)].copy()
    print(f'  Dropped {n_energy_bad + n_duration_bad} rows. Remaining: {len(df)}')

snapshot('Step 4', 'Validated Energy > 0 and Duration > 0', df)

Energy Consumed <= 0  : 0 rows
Charging Duration <= 0: 0 rows

 All Energy and Duration values are strictly positive — no action needed.


---
## Step 5 · Temperature — Plausibility Check & Clipping

Audit found temperatures as high as **73°C**, which is physically impossible for an outdoor
ambient temperature. The plausible real-world range for EV charging is **−30°C to 50°C**.
Values outside this range are sensor/data errors and are clipped.

In [13]:
temp_col = 'Temperature (C)'

TEMP_MIN, TEMP_MAX = -30, 50

n_low  = (df[temp_col] < TEMP_MIN).sum()
n_high = (df[temp_col] > TEMP_MAX).sum()

print(f'Temperature BEFORE clipping:')
print(f'  Range: [{df[temp_col].min():.2f}, {df[temp_col].max():.2f}]')
print(f'  Values < {TEMP_MIN}°C : {n_low}')
print(f'  Values > {TEMP_MAX}°C : {n_high}')

# Show out-of-range samples
oob_temp = df[(df[temp_col] < TEMP_MIN) | (df[temp_col] > TEMP_MAX)]
if len(oob_temp) > 0:
    print(f'\nSample out-of-range temperatures:')
    display(oob_temp[[temp_col, 'Charging Station Location', 'Charger Type']].head(10))

df[temp_col] = df[temp_col].clip(TEMP_MIN, TEMP_MAX)

print(f'\nTemperature AFTER clipping:')
print(f'  Range: [{df[temp_col].min():.2f}, {df[temp_col].max():.2f}]')
print(f'   All values now within [{TEMP_MIN}, {TEMP_MAX}]')

snapshot('Step 5', f'Clipped Temperature to [{TEMP_MIN}, {TEMP_MAX}]°C', df)

Temperature BEFORE clipping:
  Range: [-10.72, 73.17]
  Values < -30°C : 0
  Values > 50°C : 5

Sample out-of-range temperatures:


,Temperature (C),Charging Station Location,Charger Type
35,59.9320,New York,Level 1
183,58.1052,San Francisco,DC Fast Charger
842,50.5439,San Francisco,Level 1
851,69.4859,Los Angeles,Level 1
1045,73.1696,Houston,DC Fast Charger



Temperature AFTER clipping:
  Range: [-10.72, 50.00]
   All values now within [-30, 50]


---
## Step 6 · Duplicate Detection

In [14]:
# ── Full exact duplicates ────────────────────────────────────────────────────────
n_full_dups = df.duplicated().sum()
print(f'Exact duplicate rows: {n_full_dups}')

# ── Logical duplicates: same user, same session start time ──────────────────────
n_logical_dups = df.duplicated(subset=['User ID', 'Charging Start Time']).sum()
print(f'Logical duplicates (User ID + Start Time): {n_logical_dups}')

if n_full_dups > 0:
    print(f'\nDropping {n_full_dups} full duplicate rows...')
    df = df.drop_duplicates().copy()

if n_logical_dups > 0:
    print(f'Dropping {n_logical_dups} logical duplicates (keeping first occurrence)...')
    df = df.drop_duplicates(subset=['User ID', 'Charging Start Time'], keep='first').copy()

if n_full_dups == 0 and n_logical_dups == 0:
    print('\n No duplicates found — dataset integrity confirmed.')

snapshot('Step 6', 'Duplicate check', df)

Exact duplicate rows: 0
Logical duplicates (User ID + Start Time): 0

 No duplicates found — dataset integrity confirmed.


---
## Step 7 · Standardise Categorical Text Columns

Even though the data looks clean, production pipelines must be robust to whitespace and
inconsistent casing. We apply strip + title-case to all string/object columns.

In [15]:
CAT_COLS = [
    'Vehicle Model',
    'Charging Station Location',
    'Time of Day',
    'Day of Week',
    'Charger Type',
    'User Type'
]

print('=== Value Counts BEFORE standardisation ===')
for col in CAT_COLS:
    print(f'\n{col}: {sorted(df[col].unique())}')

# Apply strip + title-case
for col in CAT_COLS:
    df[col] = df[col].str.strip().str.title()

print('\n\n=== Value Counts AFTER standardisation ===')
for col in CAT_COLS:
    print(f'\n{col}: {sorted(df[col].unique())}')

print('\n All categorical columns standardised.')
snapshot('Step 7', 'Standardised categorical text (strip + title-case)', df)

=== Value Counts BEFORE standardisation ===

Vehicle Model: ['BMW i3', 'Chevy Bolt', 'Hyundai Kona', 'Nissan Leaf', 'Tesla Model 3']

Charging Station Location: ['Chicago', 'Houston', 'Los Angeles', 'New York', 'San Francisco']

Time of Day: ['Afternoon', 'Evening', 'Morning', 'Night']

Day of Week: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

Charger Type: ['DC Fast Charger', 'Level 1', 'Level 2']

User Type: ['Casual Driver', 'Commuter', 'Long-Distance Traveler']




=== Value Counts AFTER standardisation ===



Vehicle Model: ['Bmw I3', 'Chevy Bolt', 'Hyundai Kona', 'Nissan Leaf', 'Tesla Model 3']

Charging Station Location: ['Chicago', 'Houston', 'Los Angeles', 'New York', 'San Francisco']

Time of Day: ['Afternoon', 'Evening', 'Morning', 'Night']

Day of Week: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

Charger Type: ['Dc Fast Charger', 'Level 1', 'Level 2']

User Type: ['Casual Driver', 'Commuter', 'Long-Distance Traveler']

 All categorical columns standardised.


---
## Step 8 · Parse Datetime Columns & Add Derived Features

Parse timestamps properly and add a few lossless derived columns useful for modelling.

In [16]:
df['Charging Start Time'] = pd.to_datetime(df['Charging Start Time'], errors='coerce')
df['Charging End Time']   = pd.to_datetime(df['Charging End Time'],   errors='coerce')

# Derived time features
df['start_hour']   = df['Charging Start Time'].dt.hour
df['start_month']  = df['Charging Start Time'].dt.month
df['start_dow_num']= df['Charging Start Time'].dt.dayofweek  # 0=Mon
df['is_weekend']   = df['start_dow_num'].isin([5, 6]).astype(int)

# SoC delta (useful feature: how much did battery charge in this session?)
df['soc_delta_pp'] = df['State of Charge (End %)'] - df['State of Charge (Start %)']

n_bad_start = df['Charging Start Time'].isna().sum()
n_bad_end   = df['Charging End Time'].isna().sum()
print(f'Unparseable Start Times: {n_bad_start}')
print(f'Unparseable End Times  : {n_bad_end}')
print(f'\nNew columns added: start_hour, start_month, start_dow_num, is_weekend, soc_delta_pp')
print(f'Total columns now: {df.shape[1]}')

snapshot('Step 8', 'Parsed datetimes; added 5 derived features', df)

Unparseable Start Times: 0


Unparseable End Times  : 0

New columns added: start_hour, start_month, start_dow_num, is_weekend, soc_delta_pp
Total columns now: 26


---
## Step 9 · Final Dataset Inspection

In [17]:
print('=== Final Dataset Info ===')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Total nulls: {df.isnull().sum().sum()}')
print()

print('Column dtypes:')
dtype_summary = pd.DataFrame({
    'Dtype'  : df.dtypes.values,
    'Non-Null': df.notnull().sum().values,
    'Null'   : df.isnull().sum().values
}, index=df.columns)
display(dtype_summary)

print('\nSample (first 5 rows):')
display(df.head(5))

=== Final Dataset Info ===
Shape: 1,320 rows × 26 columns
Total nulls: 0

Column dtypes:


,Dtype,Non-Null,Null
User ID,str,1320,0
Vehicle Model,str,1320,0
Battery Capacity (kWh),float64,1320,0
Charging Station ID,str,1320,0
Charging Station Location,str,1320,0
Charging Start Time,datetime64[us],1320,0
Charging End Time,datetime64[us],1320,0
Energy Consumed (kWh),float64,1320,0
Charging Duration (hours),float64,1320,0
Charging Rate (kW),float64,1320,0



Sample (first 5 rows):


,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),Time of Day,Day of Week,State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (C),Vehicle Age (years),Charger Type,User Type,soc_was_swapped,start_hour,start_month,start_dow_num,is_weekend,soc_delta_pp
0,User_1,Bmw I3,108.4630,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.7123,0.5914,36.3892,13.0877,Evening,Tuesday,29.3716,86.1200,293.6021,27.9480,2.0000,Dc Fast Charger,Commuter,0,0,1,0,0,56.7484
1,User_2,Hyundai Kona,100.0000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.3393,3.1337,30.6777,21.1284,Morning,Monday,10.1158,84.6643,112.1128,14.3110,3.0000,Level 1,Casual Driver,0,1,1,0,0,74.5486
2,User_3,Chevy Bolt,75.0000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.1289,2.4527,27.5136,35.6673,Morning,Thursday,6.8546,69.9176,71.7993,21.0020,2.0000,Level 2,Commuter,0,2,1,0,0,63.0630
3,User_4,Hyundai Kona,50.0000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.4578,1.2664,32.8829,13.0362,Evening,Saturday,83.1200,99.6243,199.5778,38.3163,1.0000,Level 1,Long-Distance Traveler,0,3,1,0,0,16.5043
4,User_5,Hyundai Kona,50.0000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.6291,2.0198,10.2157,10.1615,Morning,Saturday,54.2589,63.7438,203.6618,-7.8342,1.0000,Level 1,Long-Distance Traveler,0,4,1,0,0,9.4848


In [18]:
# Numeric summary of key columns after cleaning
key_num_cols = [
    'Energy Consumed (kWh)', 'Charging Duration (hours)',
    'Charging Rate (kW)', 'Charging Cost (USD)',
    'State of Charge (Start %)', 'State of Charge (End %)',
    'soc_delta_pp', 'Distance Driven (since last charge) (km)',
    'Temperature (C)', 'Battery Capacity (kWh)'
]
print('Descriptive stats (cleaned data):')
display(df[key_num_cols].describe().T.rename(columns={'50%': 'median'}))

Descriptive stats (cleaned data):


,count,mean,std,min,25%,median,75%,max
Energy Consumed (kWh),1320.0000,42.6501,21.8505,0.0458,24.8681,42.2855,60.5455,152.2388
Charging Duration (hours),1320.0000,2.2694,1.0610,0.0953,1.3976,2.2581,3.1128,7.6351
Charging Rate (kW),1320.0000,25.9430,13.6572,1.4725,14.4313,25.8578,36.9558,97.3423
Charging Cost (USD),1320.0000,22.5514,10.7515,0.2343,13.3681,22.0764,31.6460,69.4077
State of Charge (Start %),1320.0000,46.0348,21.0538,2.3260,27.5220,47.7064,62.1826,97.0590
State of Charge (End %),1320.0000,77.6728,14.1336,25.0414,66.6923,79.2127,88.9925,100.0000
soc_delta_pp,1320.0000,31.6379,22.1947,0.0008,12.8555,27.0445,47.7218,92.8257
Distance Driven (since last charge) (km),1320.0000,153.4990,83.8727,0.8624,81.9316,155.7291,221.9623,398.3648
Temperature (C),1320.0000,15.2172,14.6941,-10.7248,2.8007,14.6308,27.9818,50.0000
Battery Capacity (kWh),1320.0000,74.5347,20.6269,1.5328,62.0000,75.0000,85.0000,193.0031


---
## Step 10 · Save Cleaned Dataset

In [19]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv(CLEANED_PATH, index=False)

file_size_kb = os.path.getsize(CLEANED_PATH) / 1024
print(f' Cleaned dataset saved to: {CLEANED_PATH}')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {df.shape[1]}')
print(f'   File    : {file_size_kb:.1f} KB')

 Cleaned dataset saved to: ../data/processed/ev_charging_cleaned.csv


   Rows    : 1,320
   Columns : 26
   File    : 402.7 KB


---
## Step 11 · Before / After Summary Table

In [20]:
snapshot('Final', 'Cleaned dataset ready', df)

print('=' * 70)
print('  CLEANING PIPELINE — BEFORE / AFTER SUMMARY')
print('=' * 70)
show_log()

  CLEANING PIPELINE — BEFORE / AFTER SUMMARY


,Step,Action,Rows,Nulls,Rows Δ,Nulls Δ
0,Baseline,Raw data loaded,1320,198,0,0
1,Step 1,Renamed temperature column,1320,198,0,0
2,Step 2a,Imputed Energy & Rate (Charger Type group median),1320,66,0,-132
3,Step 2b,Imputed Distance Driven (User Type group median),1320,0,0,-66
4,Step 3a,"Clipped SoC Start/End to [0, 100]",1320,0,0,0
5,Step 3b,Swapped reversed SoC pairs; added soc_was_swapped flag,1320,0,0,0
6,Step 4,Validated Energy > 0 and Duration > 0,1320,0,0,0
7,Step 5,"Clipped Temperature to [-30, 50]°C",1320,0,0,0
8,Step 6,Duplicate check,1320,0,0,0
9,Step 7,Standardised categorical text (strip + title-case),1320,0,0,0


In [21]:
# Per-column null comparison: raw vs. cleaned
raw_nulls     = df_raw.isnull().sum().rename('Raw Nulls')
cleaned_nulls = df.reindex(columns=df_raw.columns).isnull().sum().rename('Cleaned Nulls')

null_comparison = pd.concat([raw_nulls, cleaned_nulls], axis=1)
null_comparison['Resolved'] = null_comparison['Raw Nulls'] - null_comparison['Cleaned Nulls']
null_comparison = null_comparison[null_comparison['Raw Nulls'] > 0]

print('Null resolution by column:')
display(null_comparison)

Null resolution by column:

,Raw Nulls,Cleaned Nulls,Resolved
Energy Consumed (kWh),66,0,66
Charging Rate (kW),66,0,66
Distance Driven (since last charge) (km),66,0,66


---
##  Cleaning Summary — Key Decisions

| Step | Issue Found | Rows Affected | Action Taken | Rationale |
|------|-------------|---------------|-------------|----------|
| 1 | Garbled temperature column name | All rows | Renamed to `Temperature (C)` | Encoding artefact from Windows-1252 |
| 2a | 66 nulls in `Energy Consumed` & `Charging Rate` | 66 rows | Grouped median by **Charger Type** | Physics-driven grouping; median robust to skew |
| 2b | 66 nulls in `Distance Driven` | 66 rows | Grouped median by **User Type** | Commuters vs. casual drivers have different mileage patterns |
| 3a | SoC Start/End > 100% or < 0% | 9 + 23 rows | Clipped to [0, 100] | Sensor rounding errors; sessions are valid |
| 3b | SoC End < SoC Start | 268 rows | Swapped Start ↔ End; added `soc_was_swapped` flag | Preserves 20% of data; flag enables model awareness |
| 4 | Energy ≤ 0 or Duration ≤ 0 | 0 rows | No action needed | Validated clean |
| 5 | Temperature > 50°C (max 73°C found) | ~few rows | Clipped to [−30, 50]°C | Physically impossible; clip rather than drop |
| 6 | Duplicate rows / logical duplicates | 0 rows | No action needed | Dataset has unique sessions |
| 7 | Categorical inconsistency | All rows | Strip + title-case | Pipeline robustness |
| 8 | No parsed datetime features | All rows | Added 5 derived columns | Enables time-based modelling |

### Rows: 1,320 → 1,320 (zero rows dropped — all issues resolved in place)
### Nulls: 198 → 0 (100% resolved via imputation)
### New columns: 5 added (`start_hour`, `start_month`, `start_dow_num`, `is_weekend`, `soc_delta_pp`, `soc_was_swapped`)

>  Raw CSV untouched. Cleaned file at `data/processed/ev_charging_cleaned.csv`  
> ️ Proceed to **`03_exploratory_data_analysis.ipynb`**